# YOLOv8x-seg — Learning Notebook

## What is YOLOv8?

YOLO (You Only Look Once) is a family of real-time object detection models. The key idea: instead of scanning an image in multiple passes, YOLO looks at the entire image **once** and predicts all bounding boxes and class labels simultaneously. This makes it extremely fast.

**YOLOv8** is the 8th major generation (released by Ultralytics, 2023). It improves accuracy and speed over previous versions while keeping the same simple API.

---

## What does `-seg` mean?

YOLO comes in several task variants:

| Suffix | Task | Output |
|---|---|---|
| *(none)* | Object detection | Bounding boxes + class labels |
| `-seg` | **Instance segmentation** | Bounding boxes + class labels + **pixel masks** |
| `-pose` | Pose estimation | Bounding boxes + keypoints (joints) |
| `-cls` | Image classification | Single class label for the whole image |

**Instance segmentation** goes one step further than detection: instead of just drawing a box around each object, it outlines the exact pixels that belong to each object. This means you can tell precisely which pixels are "knife" vs "background".

```
Detection:          Segmentation:
┌─────────┐         ░░░░░░░░░░░
│  knife  │         ░░▓▓▓▓▓░░░
│         │   vs    ░░▓▓▓▓▓▓░░
│         │         ░░░░▓▓▓▓░░
└─────────┘         ░░░░░░░░░░░
  (box)               (mask)
```

In this project, segmentation masks allow **pixel-level contact detection** — we can check whether a person's mask and a knife's mask share any pixel, which is far more accurate than just checking if their boxes overlap.

---

## What does `x` mean?

YOLO models come in 5 sizes. Larger = more accurate but slower:

| Variant | Parameters | Speed | mAP (COCO) | Use case |
|---|---|---|---|---|
| `n` (nano)   | 3.4M  | fastest | 36.7 | Mobile / edge devices |
| `s` (small)  | 11.8M | fast    | 44.9 | Raspberry Pi, webcam |
| `m` (medium) | 25.9M | medium  | 52.9 | Good default for most tasks |
| `l` (large)  | 43.7M | slow    | 53.9 | Server inference |
| `x` (extra)  | 68.2M | slowest | **54.7** | Maximum accuracy — what we use |

We use `x` because this is a safety-critical application (detecting dangerous items near children) and we're running on Apple Silicon (MPS), which handles large models well. A missed detection is worse than a slightly slower FPS.

---

## What is COCO?

COCO (Common Objects in Context) is the benchmark dataset YOLOv8x-seg was trained on. It contains ~330,000 images with 80 object categories — everyday items like people, cars, animals, furniture, and kitchen objects.

The model "knows" these 80 classes out of the box. For anything outside COCO (like lighters or pill bottles), you need either open-vocabulary detection (YOLOWorld) or fine-tuning.

---

## What is mAP?

**mAP** (mean Average Precision) is the standard accuracy metric for object detection.

- **Precision** = of all the detections the model made, what fraction were correct?
- **Recall** = of all the real objects in the image, what fraction did the model find?
- **AP** = the area under the precision-recall curve for one class
- **mAP** = the average AP across all classes

**mAP50** means IoU threshold = 0.5 (a box counts as correct if it overlaps the ground truth by at least 50%).  
**mAP50-95** averages across IoU thresholds from 0.5 to 0.95 — a stricter metric.

---

## What is IoU?

**IoU** (Intersection over Union) measures how much two bounding boxes overlap:

```
IoU = Area of Overlap / Area of Union

IoU = 0.0  →  boxes don't touch at all
IoU = 0.5  →  boxes overlap by 50%
IoU = 1.0  →  boxes are identical
```

IoU is used in two places:
1. **NMS (Non-Maximum Suppression)** — if two detections of the same class overlap by more than `iou` threshold, the weaker one is deleted. This prevents the model from reporting the same knife 5 times.
2. **Evaluation** — deciding whether a predicted box is "close enough" to the ground truth to count as correct.

---
## Part 1 — Setup

In [ ]:
# Install if needed:
# pip install ultralytics opencv-python numpy matplotlib

import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

print("Imports OK")

---
## Part 2 — Load the Model

Loading is one line. On first run, Ultralytics automatically downloads `yolov8x-seg.pt` (~130 MB) from the official hub.

After loading, `model.names` is a dictionary mapping class ID → class name, covering all 80 COCO categories.

In [ ]:
model = YOLO("yolov8x-seg.pt")

# Print all 80 COCO classes the model knows
print(f"Model knows {len(model.names)} classes:\n")
for id_, name in model.names.items():
    print(f"  {id_:>3}  {name}")

---
## Part 3 — Key Parameters (read this before running inference)

### `conf` — Confidence threshold

Every detection has a **confidence score** between 0 and 1 — how sure the model is that this detection is real. Detections below `conf` are discarded.

```
conf = 0.25  →  keep detections where model is ≥25% confident  (default, more detections, more false positives)
conf = 0.50  →  keep only detections where model is ≥50% confident (fewer detections, fewer false positives)
conf = 0.80  →  very strict — only very obvious objects
```

**Rule of thumb:**
- Trained model (like YOLOv8x-seg on COCO) → start with `0.25`
- Safety-critical application (don't want to miss things) → go lower, e.g. `0.15`
- Too many false positives → go higher, e.g. `0.40`

### `iou` — IoU threshold for NMS

Controls how aggressively duplicate detections are suppressed. If two boxes for the same class overlap by more than `iou`, the weaker one is deleted.

```
iou = 0.3  →  aggressive suppression — overlapping objects may be merged into one
iou = 0.5  →  standard (default)
iou = 0.7  →  permissive — more boxes kept, useful when objects genuinely overlap
```

### `imgsz` — Inference image size

The image is resized to `imgsz × imgsz` before being fed to the model (letterboxed to preserve aspect ratio). Larger = better detail but slower.

```
imgsz = 320  →  fastest, misses small/distant objects
imgsz = 640  →  default, good balance
imgsz = 1280 →  much better at small objects, ~2× slower
```

**Key insight:** An object that is 15×15 pixels in a 1280×720 camera frame appears as ~30×30 pixels when the frame is resized to 640, but only ~7×7 pixels at 320. Below ~10 pixels, YOLO reliably fails. If you need to detect small distant objects, increase `imgsz`.

### `device` — Hardware backend

```python
device = "mps"   # Apple Silicon GPU (M1/M2/M3) — use this on Mac
device = "cuda"  # NVIDIA GPU
device = "cpu"   # CPU fallback (slow, ~5-10× slower than GPU)
device = 0       # First NVIDIA GPU (same as "cuda:0")
```

### `classes` — Filter by class ID

Only return detections for specific COCO class IDs. Useful when you don't care about most classes.

```python
classes = [0]        # only detect persons
classes = [0, 43]    # persons and knives
classes = None       # detect all 80 classes (default)
```

### `verbose` — Suppress output

```python
verbose = True   # print inference speed and result count each frame (default)
verbose = False  # silent — use this in loops to avoid console spam
```

---
## Part 4 — Run Inference on a Single Image

### 4a. Load a test image

We'll grab a sample image from the internet, or you can point to a local file.

In [ ]:
# Option A: use Ultralytics' built-in test image (downloaded automatically)
from ultralytics.utils import ASSETS
img_path = str(ASSETS / "bus.jpg")   # a bus + pedestrians photo

# Option B: use your own image
# img_path = "/path/to/your/image.jpg"

img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)   # matplotlib expects RGB

print(f"Image shape: {img.shape}  (height, width, channels)")
plt.figure(figsize=(10, 6))
plt.imshow(img_rgb)
plt.title("Input image")
plt.axis("off")
plt.show()

### 4b. Run inference

`.predict()` accepts: file path, NumPy array, URL, PIL Image, or a list of any of these.

In [ ]:
results = model.predict(
    source=img_path,
    conf=0.25,       # confidence threshold
    iou=0.45,        # NMS IoU threshold
    imgsz=640,       # inference resolution
    device="mps",    # change to "cpu" if not on Apple Silicon
    classes=None,    # detect all classes
    verbose=True,    # print inference time
)

# results is a list — one element per input image
r = results[0]
print(f"\nType of results[0]: {type(r)}")

---
## Part 5 — Understanding the Result Object

The result object `r` contains everything the model found. Let's explore each part.

In [ ]:
# ── r.boxes — Bounding box detections ─────────────────────────────────────────
print("=== r.boxes ===")
print(f"Number of detections: {len(r.boxes)}\n")

for i, box in enumerate(r.boxes):
    cls_id   = int(box.cls[0].item())       # class ID (integer, 0-79)
    cls_name = model.names[cls_id]           # class name string
    conf     = float(box.conf[0].item())     # confidence score (0.0-1.0)

    # Bounding box coordinates — two formats:
    xyxy = box.xyxy[0].tolist()   # [x1, y1, x2, y2] — top-left and bottom-right corners
    xywh = box.xywh[0].tolist()   # [cx, cy, w, h]   — centre + width/height

    # Track ID — only available when using .track(), not .predict()
    track_id = int(box.id[0].item()) if box.id is not None else None

    print(f"  [{i}] {cls_name:<15} conf={conf:.2f}  xyxy={[round(v) for v in xyxy]}  id={track_id}")

In [ ]:
# ── r.masks — Segmentation masks ──────────────────────────────────────────────
print("=== r.masks ===")

if r.masks is None:
    print("No masks — this happens if no objects were detected, or if using a non-seg model.")
else:
    print(f"r.masks.data shape: {r.masks.data.shape}")
    # Shape is (N, H, W) where:
    #   N = number of detections
    #   H, W = mask resolution (smaller than the original image — typically 160×160)
    # Each mask is a float tensor: values close to 1 = object pixel, close to 0 = background

    print(f"Number of masks: {r.masks.data.shape[0]}")
    print(f"Mask resolution: {r.masks.data.shape[1]} × {r.masks.data.shape[2]}")
    print(f"Value range: [{r.masks.data.min():.2f}, {r.masks.data.max():.2f}]")
    print()
    print("To get a usable boolean mask at full image resolution:")
    print("  raw = r.masks.data[i].cpu().numpy()         # float (H_mask, W_mask)")
    print("  mask = cv2.resize(raw, (img_w, img_h)) > 0.5  # bool (img_h, img_w)")

In [ ]:
# ── r.orig_img — Original image ────────────────────────────────────────────────
print(f"r.orig_img shape: {r.orig_img.shape}  (the input image as a NumPy array, BGR)")
print(f"r.orig_shape:     {r.orig_shape}       (original H, W before any resizing)")

# ── r.speed — Inference timing ─────────────────────────────────────────────────
print(f"\nr.speed: {r.speed}")
# Keys: 'preprocess' (resize+normalise), 'inference' (model forward pass), 'postprocess' (NMS+mask decoding)

---
## Part 6 — Visualise Results

### 6a. Using `.plot()` — the easiest way

`.plot()` returns a BGR NumPy array with all detections drawn (boxes, masks, labels).

In [ ]:
annotated = r.plot()   # returns BGR image with all detections rendered

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title("r.plot() — boxes + masks + labels")
plt.axis("off")
plt.show()

# plot() options:
# r.plot(conf=True)    # show confidence scores (default True)
# r.plot(boxes=False)  # hide bounding boxes, show masks only
# r.plot(masks=False)  # hide masks, show boxes only
# r.plot(labels=False) # hide class labels

### 6b. Visualising individual masks manually

Sometimes you need to work with a specific object's mask directly — for example, to check overlap with another object.

In [ ]:
if r.masks is not None and len(r.masks.data) > 0:
    img_h, img_w = r.orig_shape
    masks_data   = r.masks.data.cpu().numpy()   # shape: (N, mask_h, mask_w)

    # Show the first 4 masks (or fewer if less detected)
    n_show = min(4, len(masks_data))
    fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4))
    if n_show == 1:
        axes = [axes]

    for i in range(n_show):
        raw_mask  = masks_data[i]                                    # float (mask_h, mask_w)
        full_mask = cv2.resize(raw_mask, (img_w, img_h))             # float (img_h, img_w)
        bool_mask = full_mask > 0.5                                  # boolean

        cls_name = model.names[int(r.boxes[i].cls[0].item())]
        conf     = float(r.boxes[i].conf[0].item())

        axes[i].imshow(bool_mask, cmap="Blues")
        axes[i].set_title(f"{cls_name} ({conf:.0%})\n"
                          f"pixels: {bool_mask.sum():,}")
        axes[i].axis("off")

    plt.suptitle("Individual segmentation masks (resized to image resolution)")
    plt.tight_layout()
    plt.show()
else:
    print("No masks to display.")

### 6c. Overlay a mask on the original image

In [ ]:
if r.masks is not None and len(r.masks.data) > 0:
    img_h, img_w = r.orig_shape
    orig = r.orig_img.copy()   # BGR

    # Overlay the first detection's mask in semi-transparent red
    raw_mask  = r.masks.data[0].cpu().numpy()
    bool_mask = cv2.resize(raw_mask, (img_w, img_h)) > 0.5

    overlay = orig.copy()
    overlay[bool_mask] = [0, 0, 200]    # paint mask pixels red (BGR)
    blended = cv2.addWeighted(orig, 0.6, overlay, 0.4, 0)   # 40% opacity

    cls_name = model.names[int(r.boxes[0].cls[0].item())]

    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(blended, cv2.COLOR_BGR2RGB))
    plt.title(f"Mask overlay for: {cls_name}")
    plt.axis("off")
    plt.show()

---
## Part 7 — `.track()` vs `.predict()`

### `.predict()` — single frame, no memory

Runs detection independently on each image/frame. Every call is stateless — the model has no idea what it saw last frame. Good for single images or when you don't need consistent IDs.

### `.track()` — multi-frame, with memory

Adds a **tracker** on top of detection. The tracker links detections across frames so each person/object gets a consistent ID that doesn't change frame-to-frame.

Without tracking:
```
Frame 1: person A detected (no ID)
Frame 2: person A detected (no ID) — model doesn't know it's the same person
```

With tracking:
```
Frame 1: person detected → assigned ID=1
Frame 2: same person detected → still ID=1  ← consistent!
Frame 3: a new person enters → assigned ID=2
```

In our project, `.track()` is used in the main demo so the same child keeps the same ID even as they move. The benchmark uses `.predict()` because we don't need tracking — we just want to know whether the object is detected.

In [ ]:
# Demonstrate .track() on a static image (no real tracking without a video stream)

track_results = model.track(
    source=img_path,
    persist=True,    # keep track state between calls — MUST be True in a video loop
    conf=0.25,
    iou=0.45,
    device="mps",
    verbose=False,
)

r_track = track_results[0]
print("Track IDs assigned:")
for box in r_track.boxes:
    cls_name = model.names[int(box.cls[0].item())]
    track_id = int(box.id[0].item()) if box.id is not None else "(no ID)"
    conf     = float(box.conf[0].item())
    print(f"  {cls_name:<15} conf={conf:.2f}  track_id={track_id}")

---
## Part 8 — Filter by Specific Classes

In [ ]:
# Only detect persons (class 0) and buses (class 5)
filtered_results = model.predict(
    source=img_path,
    classes=[0, 5],   # person=0, bus=5
    conf=0.25,
    device="mps",
    verbose=False,
)

r_filtered = filtered_results[0]
print(f"Detections with classes=[0, 5]: {len(r_filtered.boxes)}")
for box in r_filtered.boxes:
    print(f"  {model.names[int(box.cls[0].item())]}  conf={float(box.conf[0].item()):.2f}")

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(r_filtered.plot(), cv2.COLOR_BGR2RGB))
plt.title("Filtered: persons and buses only")
plt.axis("off")
plt.show()

---
## Part 9 — Confidence Threshold Comparison

See how changing `conf` affects how many detections you get.

In [ ]:
conf_values = [0.10, 0.25, 0.50, 0.75]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, conf_val in zip(axes, conf_values):
    res = model.predict(
        source=img_path,
        conf=conf_val,
        device="mps",
        verbose=False,
    )
    n = len(res[0].boxes)
    ax.imshow(cv2.cvtColor(res[0].plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(f"conf={conf_val}\n{n} detections")
    ax.axis("off")

plt.suptitle("Effect of confidence threshold on number of detections", fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 10 — imgsz Comparison

See how inference resolution affects detection quality and speed.

In [ ]:
import time

sizes = [320, 640, 1280]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, sz in zip(axes, sizes):
    t0  = time.time()
    res = model.predict(source=img_path, imgsz=sz, conf=0.25, device="mps", verbose=False)
    ms  = (time.time() - t0) * 1000
    n   = len(res[0].boxes)
    ax.imshow(cv2.cvtColor(res[0].plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(f"imgsz={sz}\n{n} detections | {ms:.0f} ms")
    ax.axis("off")

plt.suptitle("Effect of imgsz on detection count and inference speed", fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 11 — Pixel-Level Mask Overlap (the core technique in our project)

This is the technique used in `yoloworld_demo.py` to detect contact between a person and a dangerous object.

In [ ]:
# Simulate: do any two detected objects' masks overlap?

if r.masks is not None and len(r.masks.data) >= 2:
    img_h, img_w = r.orig_shape
    masks_data   = r.masks.data.cpu().numpy()

    def get_bool_mask(raw, w, h):
        return cv2.resize(raw, (w, h)) > 0.5

    mask_0 = get_bool_mask(masks_data[0], img_w, img_h)
    mask_1 = get_bool_mask(masks_data[1], img_w, img_h)

    overlap = np.logical_and(mask_0, mask_1)
    touching = overlap.any()

    name_0 = model.names[int(r.boxes[0].cls[0].item())]
    name_1 = model.names[int(r.boxes[1].cls[0].item())]

    print(f"Object 0: {name_0}  — pixels: {mask_0.sum():,}")
    print(f"Object 1: {name_1}  — pixels: {mask_1.sum():,}")
    print(f"Overlap pixels: {overlap.sum():,}")
    print(f"Touching (any overlap): {touching}")

    # Visualise
    vis = np.zeros((img_h, img_w, 3), dtype=np.uint8)
    vis[mask_0]  = [200, 80, 80]    # red for object 0
    vis[mask_1]  = [80, 80, 200]    # blue for object 1
    vis[overlap] = [255, 255, 0]    # yellow where they overlap

    plt.figure(figsize=(10, 6))
    plt.imshow(vis)
    plt.title(f"Red={name_0}  Blue={name_1}  Yellow=overlap  |  Touching={touching}")
    plt.axis("off")
    plt.show()
else:
    print("Need at least 2 detections to demonstrate overlap. Try with a different image.")

---
## Summary — Key Takeaways

| Concept | Rule of thumb |
|---|---|
| Model size | Use `x` for accuracy, `n/s` for speed/edge devices |
| `conf` | Start at 0.25; lower it if you're missing detections, raise it if too many false positives |
| `iou` | Default 0.45 is fine; raise to 0.7 if objects genuinely overlap and get merged |
| `imgsz` | 640 for general use; 1280 if small/distant objects matter |
| `device` | Always use `"mps"` on Mac, `"cuda"` on NVIDIA — CPU is ~10× slower |
| `.predict()` vs `.track()` | Use `.track(persist=True)` in video loops; `.predict()` for single images or benchmarks |
| Masks | Always resize from model resolution to image resolution with `cv2.resize()` before use |
| Overlap detection | `np.logical_and(mask_a, mask_b).any()` — one line, pixel-perfect |

**What YOLOv8x-seg cannot do:**
- Detect objects outside the 80 COCO classes (lighters, pill bottles, etc.) → use YOLOWorld or fine-tune
- Reliably detect very small objects (< ~15 px) → increase `imgsz` or use SAHI
- Distinguish between visually similar objects (regular bottle vs. pill bottle) → need fine-tuning with specific training data